In [1]:
import pandas as pd
import urllib.request
import zipfile

url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
urllib.request.urlretrieve(url, "bank.zip")

with zipfile.ZipFile("bank.zip", "r") as z:
    z.extractall("data")

with zipfile.ZipFile("data/bank-additional.zip", "r") as z:
    z.extractall("data")

print("✅ Downloaded and extracted")

✅ Downloaded and extracted


In [2]:
import boto3

df = pd.read_csv("data/bank-additional/bank-additional-full.csv", sep=";")
print("Shape:", df.shape)
display(df.head())

bucket = "mlops-sagemaker-nirmai-v1"
df.to_csv("bank.csv", index=False)
boto3.client("s3").upload_file("bank.csv", bucket, "raw/bank.csv")
print(f"✅ Uploaded to s3://{bucket}/raw/bank.csv")

✅ Uploaded to s3://mlops-sagemaker-nirmai-v1/raw/bank.csv


In [3]:
!pip install "sagemaker[local]"==2.224.0 -q

In [4]:
import sagemaker
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

bucket = "mlops-sagemaker-nirmai-v1"
session = sagemaker.Session()
role = sagemaker.get_execution_role()

processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.t3.medium",
    instance_count=1
)

processor.run(
    code="src/preprocessing.py",
    inputs=[
        ProcessingInput(
            source=f"s3://{bucket}/raw/bank.csv",
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train",
            destination=f"s3://{bucket}/processed/train"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test",
            destination=f"s3://{bucket}/processed/test"
        )
    ],
    arguments=[
        "--input-data", "/opt/ml/processing/input/bank.csv",
        "--output-train", "/opt/ml/processing/output/train",
        "--output-test", "/opt/ml/processing/output/test"
    ]
)
print("✅ Processing job complete")

INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-2026-06-06-08-41-57-594


.

.

.

.

...

.

.

.

📥 Reading data...
⚙️ Preprocessing...
✂️ Splitting data...
✅ Done!


✅ Processing job complete


In [5]:
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.session import Session
import boto3
import time

region = "us-east-1"
boto_session = boto3.Session(region_name=region)
sagemaker_session = Session(boto_session=boto_session)
role = sagemaker.get_execution_role()
bucket = "mlops-sagemaker-nirmai-v1"

df = pd.read_csv(f"s3://{bucket}/processed/train/train.csv")

# Add required Feature Store columns
df["event_time"] = float(time.time())
df["record_id"] = df.index.astype(str)

# Clean column names
df.columns = (
    df.columns
    .str.replace(".", "_", regex=False)
    .str.replace(" ", "_")
    .str.replace(r"[^a-zA-Z0-9_-]", "", regex=True)
    .str.strip("_")
)

feature_group_name = "bank-term-deposit-features"
feature_group = FeatureGroup(
    name=feature_group_name,
    sagemaker_session=sagemaker_session
)
feature_group.load_feature_definitions(data_frame=df)
feature_group.create(
    s3_uri=f"s3://{bucket}/feature-store",
    record_identifier_name="record_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=False
)
print("Feature group creation started...")

feature_group.ingest(data_frame=df, max_workers=3, wait=True)
print("✅ Feature ingestion complete")

ResourceInUse: An error occurred (ResourceInUse) when calling the CreateFeatureGroup operation: Resource Already Exists: FeatureGroup with name bank-term-deposit-features already exists. Choose a different name.

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib, os, tarfile, shutil

# Load from feature store (update path to your account/region)
df = pd.read_parquet(
    "s3://mlops-sagemaker-nirmai-v1/feature-store/928747699677/sagemaker/us-east-1/offline-store/bank-term-deposit-features-1780647667/data/"
)

# Drop Feature Store metadata columns
df = df.drop(columns=["record_id", "event_time"])
df = df.drop(columns=df.select_dtypes(include=["datetime64"]).columns)
for col in ["write_time", "api_invocation_time"]:
    df[col] = pd.to_datetime(df[col]).astype("int64") // 10**9
X = df.drop("y", axis=1)
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# One-hot encode — must use drop_first=True consistently
X_train = pd.get_dummies(X_train, drop_first=True)
X_test  = pd.get_dummies(X_test,  drop_first=True)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

scale_pos_weight: 7.89


              precision    recall  f1-score   support

           0       0.95      0.86      0.90      5849
           1       0.37      0.62      0.46       741

    accuracy                           0.84      6590
   macro avg       0.66      0.74      0.68      6590
weighted avg       0.88      0.84      0.85      6590

ROC-AUC: 0.7827258151560103


In [7]:
import os, shutil, tarfile, joblib

os.makedirs("model", exist_ok=True)

# Native XGBoost format — no joblib, no dependency issues
model.save_model("model/xgboost-model")

# feature columns still use joblib (no xgboost dependency here)
joblib.dump(list(X_train.columns), "model/feature_columns.pkl")
shutil.copy("inference.py", "model/inference.py")

print("Saved:", os.listdir("model"))

Saved: ['model.joblib', 'inference.py', 'feature_columns.pkl', 'xgboost-model']


/opt/conda/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [08:50:46] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1744329020674/work/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


In [8]:
import boto3

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model/xgboost-model",       arcname="xgboost-model")
    tar.add("model/feature_columns.pkl", arcname="feature_columns.pkl")
    tar.add("model/inference.py",        arcname="inference.py")

with tarfile.open("model.tar.gz", "r:gz") as tar:
    print("Archive contents:", tar.getnames())

bucket = "mlops-sagemaker-nirmai-v1"
boto3.client("s3").upload_file("model.tar.gz", bucket, "model/model.tar.gz")
print("✅ Uploaded")

Archive contents: ['xgboost-model', 'feature_columns.pkl', 'inference.py']


In [9]:
import sagemaker
from sagemaker.xgboost.model import XGBoostModel

role   = sagemaker.get_execution_role()
bucket = "mlops-sagemaker-nirmai-v1"

xgb_model = XGBoostModel(
    model_data=f"s3://{bucket}/model/model.tar.gz",
    role=role,
    entry_point="inference.py",
    framework_version="1.7-1"
)

predictor = xgb_model.deploy(
    instance_type="ml.m5.large",
    initial_instance_count=1
)
print("✅ Endpoint deployed:", predictor.endpoint_name)

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-06-06-08-50-47-613


INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-2026-06-06-08-50-48-253


In [ ]:
import io

# Sample row (20 features, no header, no target column)
sample = "56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0"

predictor.serializer   = sagemaker.serializers.CSVSerializer()
predictor.deserializer = sagemaker.deserializers.JSONDeserializer()

result = predictor.predict(sample)
print("Prediction:", result)


In [ ]:
predictor.delete_endpoint()
print("✅ Endpoint deleted")

In [ ]:
import boto3

sm = boto3.client("sagemaker")

sm.delete_endpoint_config(
    EndpointConfigName=predictor.endpoint_name
)

In [ ]:
import boto3

s3 = boto3.resource("s3")
bucket = s3.Bucket("mlops-sagemaker-nirmai-v1")

for obj in bucket.objects.filter(Prefix="temp/"):
    obj.delete()

In [ ]:
sm.list_endpoints()
sm.list_models()